# AI-BASED NIDS - Phase 3: Layer 2 Anomaly Detection (Keras)

This notebook trains two models on **normal network traffic**:
1. **Autoencoder (Deep Learning - Keras)**: Learns compression/decompression of normal patterns. High reconstruction error = Attack.
2. **Isolation Forest (Machine Learning)**: Tree-based outlier detection.

**Goal**: Detect unknown/zero-day attacks.

In [ ]:
# 1. Setup & Imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

In [ ]:
# 2. Load Data (Simulated or Real)
# In the real flow, you would import DataLoader from loader.py
# from data_pipeline.loader import DataLoader
# loader = DataLoader('g:/AI-BASED-NIDS/dataset')
# full_df = loader.load_files('Benign*') # Load only benign for training autoencoder

# For this notebook, we simulate scaled normal data
# 78 features, 10000 samples
input_dim = 78
num_samples = 10000
X_train_scaled = np.random.rand(num_samples, input_dim) 

print("Training Data Shape:", X_train_scaled.shape)

In [ ]:
# 3. Build Autoencoder Model (Keras)
def build_autoencoder(input_dim):
    input_layer = layers.Input(shape=(input_dim,))
    
    # Encoder
    encoded = layers.Dense(64, activation='relu')(input_layer)
    encoded = layers.Dense(32, activation='relu')(encoded)
    encoded = layers.Dense(16, activation='relu')(encoded)
    
    # Decoder
    decoded = layers.Dense(32, activation='relu')(encoded)
    decoded = layers.Dense(64, activation='relu')(decoded)
    decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)
    
    autoencoder = keras.Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

autoencoder = build_autoencoder(input_dim)
autoencoder.summary()

In [ ]:
# 4. Train Autoencoder
history = autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=20,
    batch_size=32,
    shuffle=True,
    validation_split=0.2
)

In [ ]:
# 5. Evaluation & Thresholding
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.show()

# Calculate MSE on training data to find threshold
reconstructions = autoencoder.predict(X_train_scaled)
mse = np.mean(np.power(X_train_scaled - reconstructions, 2), axis=1)
threshold = np.percentile(mse, 99) # 99th percentile as threshold
print(f"Anomaly Threshold (MSE > {threshold:.4f} is Anomaly)")

In [ ]:
# 6. Extract Encoder (Optional) & Save
autoencoder.save('autoencoder.h5')
print("Autoencoder saved to autoencoder.h5")

In [ ]:
# 7. Train Isolation Forest
clf = IsolationForest(contamination=0.01, random_state=42)
clf.fit(X_train_scaled)
joblib.dump(clf, 'isolation_forest.joblib')
print("Isolation Forest saved to isolation_forest.joblib")